# Fine-Tuning BERT for Text Classification
**Data Science Internship – February 2026 | Assignment NLP-4**

**Dataset:** IMDb Movie Reviews | **Model:** `bert-base-uncased`

> ⚡ **Recommended:** Run on Google Colab with GPU enabled
> Runtime → Change runtime type → T4 GPU

---
**Pipeline:**
```
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Comparison
```

**Experiments:**
- Experiment 1 — Freeze all BERT layers, train classifier only
- Experiment 2 — Fine-tune last 2 BERT layers + classifier
- Experiment 3 — Full BERT fine-tuning (all layers)
- Bonus — DistilBERT with learning rate scheduler + early stopping

---
## Cell 1 — Install & Import Libraries

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'transformers', 'torch', 'scikit-learn',
    'pandas', 'numpy', 'matplotlib', 'seaborn', '--quiet'
], check=True)

# Standard
import re, os, random, warnings, time
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim      import AdamW

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device     : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
print('All libraries imported successfully.')

---
## Cell 2 — Load Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTION A: Use full Kaggle IMDb dataset (recommended for real results)
#   1. Download: kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
#   2. Upload 'IMDB Dataset.csv' to Colab files panel
#   3. Uncomment the 3 lines below:
# df_full = pd.read_csv('IMDB Dataset.csv')
# df_full.columns = ['text', 'label']
# df_full['label'] = df_full['label'].map({'positive': 1, 'negative': 0})
# df = df_full.sample(n=2000, random_state=SEED).reset_index(drop=True)
# ─────────────────────────────────────────────────────────────────────────────

# OPTION B: Built-in 40-sample dataset — no download needed
pos = [
    'This film was absolutely wonderful. Performances were outstanding throughout.',
    'A masterpiece of modern cinema. Brilliant direction and a gripping storyline.',
    'I loved every minute of this movie. The cast was superb and story truly moving.',
    'One of the best films I have seen. Emotional powerful and beautifully crafted.',
    'Stunning visuals and a deeply touching story. This film will stay with me forever.',
    'Incredible acting and a script that keeps you engaged from start to finish.',
    'Highly recommended! A feel-good film with heart and wonderful performances.',
    'Phenomenal storytelling. Every scene had purpose and the acting was top notch.',
    'Riveting cinema. Kept me on the edge of my seat throughout. Loved every second.',
    'A landmark film. Visually stunning emotionally resonant and perfectly directed.',
    'Exceptional film with great character development and a truly satisfying ending.',
    'Outstanding in every way. The director has created something truly special here.',
    'Brilliant script and superb acting. This is cinema at its absolute finest.',
    'A joyful heartwarming film that leaves you feeling uplifted and deeply satisfied.',
    'Genuinely moving and beautifully shot. One of the standout films of the decade.',
    'The lead performance is extraordinary. A career best turn that deserves awards.',
    'Perfect pacing wonderful performances and a story that resonates long after.',
    'An absolute triumph. Everything about this film works from score to acting.',
    'Compelling and thought-provoking. This film challenges and rewards its audience.',
    'Delightful entertainment that manages to be both funny and genuinely emotional.',
]
neg = [
    'This was a complete waste of time. Acting was terrible and plot made no sense.',
    'Awful film. Badly written poorly directed and performances were unconvincing.',
    'I fell asleep halfway through. Boring predictable and utterly forgettable.',
    'Terrible in every way. Script was dreadful and characters flat and dull.',
    'One of the worst films I have ever seen. Do not waste your money on this.',
    'Painfully bad acting combined with a nonsensical plot makes this unwatchable.',
    'A complete disaster. No redeeming qualities whatsoever. Avoid at all costs.',
    'The direction was confused and the story went nowhere. Deeply disappointing.',
    'Dreadful from start to finish. Poor production and zero character development.',
    'Utterly pointless film. Bad dialogue bad acting and a plot full of huge holes.',
    'Horrendous. I cannot believe this got made. Every element of this film fails.',
    'Disappointing and frustrating. The source material deserved so much better.',
    'Embarrassingly bad. The script reads like it was written in an afternoon.',
    'Tedious and unpleasant. Nothing happens for two hours and the ending is awful.',
    'A cynical cash grab with no artistic merit. Insulting to the audience.',
    'Worst film of the year without question. An endurance test from beginning to end.',
    'Incoherent and boring. The director clearly had no vision for this project.',
    'Abysmal performances across the board. Not a single convincing moment in the film.',
    'A joyless slog. Poorly edited badly acted and completely devoid of any emotion.',
    'Catastrophically bad. Every decision made in this production was the wrong one.',
]

df = pd.DataFrame({'text': pos + neg, 'label': [1]*20 + [0]*20})

print(f'Dataset shape      : {df.shape}')
print(f'Missing values     : {df.isnull().sum().sum()}')
print(f'Class distribution :')
print(df['label'].value_counts().rename({1:'Positive',0:'Negative'}).to_string())
df.head(3)

---
## Cell 3 — Exploratory Data Analysis

In [ ]:
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class distribution bar chart
counts = df['label'].value_counts()
axes[0].bar(['Negative','Positive'], counts[[0,1]].values,
            color=['#b7410e','#2d6a4f'], edgecolor='white', width=0.45)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
for i, v in enumerate(counts[[0,1]].values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Word count distribution
axes[1].hist(df[df['label']==1]['word_count'], bins=12, alpha=0.75,
             color='#2d6a4f', label='Positive', edgecolor='white')
axes[1].hist(df[df['label']==0]['word_count'], bins=12, alpha=0.75,
             color='#b7410e', label='Negative', edgecolor='white')
axes[1].set_title('Word Count per Review', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Exploratory Data Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Avg word count : {df["word_count"].mean():.1f}')
print(f'Max word count : {df["word_count"].max()}')
print(f'Min word count : {df["word_count"].min()}')

---
## Cell 4 — Text Preprocessing

In [ ]:
def clean_text(text):
    """
    Light text cleaning before BERT tokenization.
    BERT handles most normalization internally via its own tokenizer,
    so we only remove noise that could confuse the tokenizer.
    Steps:
      1. Remove URLs
      2. Remove HTML tags
      3. Remove non-ASCII characters (emojis etc.)
      4. Collapse extra whitespace
    """
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', ' ', text)  # remove URLs
    text = re.sub(r'<.*?>',              ' ', text)  # remove HTML tags
    text = re.sub(r'[^\x00-\x7F]+',    ' ', text)  # remove non-ASCII
    text = re.sub(r'\s+',               ' ', text)  # collapse whitespace
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)

# Drop empty rows after cleaning
before = len(df)
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f'Rows before cleaning : {before}')
print(f'Rows after  cleaning : {len(df)}')
print(f'Rows dropped         : {before - len(df)}')

print('\nBefore vs After (2 samples):')
for i in [0, 20]:
    print(f'[{"POS" if df["label"][i]==1 else "NEG"}] ORIGINAL : {df["text"][i][:70]}')
    print(f'      CLEANED  : {df["clean_text"][i][:70]}')
    print()

---
## Cell 5 — Data Splitting (Train / Validation / Test)

In [ ]:
X = df['clean_text'].values
y = df['label'].values

# 70% train | 15% validation | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'Total samples : {len(X)}')
print(f'Train         : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Validation    : {len(X_val)}  ({len(X_val)/len(X)*100:.0f}%)')
print(f'Test          : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTrain label dist : {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Val   label dist : {dict(zip(*np.unique(y_val,   return_counts=True)))}')
print(f'Test  label dist : {dict(zip(*np.unique(y_test,  return_counts=True)))}')

---
## Cell 6 — BERT Tokenization

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128   # BERT max is 512; 128 balances speed and coverage

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded successfully.')

# Demonstrate tokenization on a sample
sample_text = 'This movie was absolutely wonderful and I loved every second of it!'
tokens      = tokenizer.tokenize(sample_text)
encoding    = tokenizer.encode_plus(
    sample_text,
    max_length            = 32,
    padding               = 'max_length',
    truncation            = True,
    return_attention_mask = True,
    return_tensors        = 'pt'
)
print(f'\nSample          : {sample_text}')
print(f'Tokens          : {tokens}')
print(f'Total tokens    : {len(tokens)}')
print(f'Input IDs       : {encoding["input_ids"][0][:12].tolist()} ...')
print(f'Attention Mask  : {encoding["attention_mask"][0][:12].tolist()} ...')
print(f'\nSpecial tokens  :')
print(f'  [CLS] id = {tokenizer.cls_token_id}   (added at start of every sequence)')
print(f'  [SEP] id = {tokenizer.sep_token_id}   (added at end of every sequence)')
print(f'  [PAD] id = {tokenizer.pad_token_id}   (fills sequences shorter than MAX_LEN)')

---
## Cell 7 — PyTorch Dataset & DataLoader

In [ ]:
class SentimentDataset(Dataset):
    """
    Custom PyTorch Dataset.
    Wraps tokenization so the DataLoader can batch samples automatically.
    Each item returns: input_ids, attention_mask, label.
    """
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer.encode_plus(
            self.texts[idx],
            max_length            = self.max_len,
            padding               = 'max_length',
            truncation            = True,
            return_attention_mask = True,
            return_tensors        = 'pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }


BATCH_SIZE = 8

train_ds = SentimentDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = SentimentDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = SentimentDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

sample_batch = next(iter(train_loader))
print(f'\nSample batch shapes:')
print(f'  input_ids      : {sample_batch["input_ids"].shape}')
print(f'  attention_mask : {sample_batch["attention_mask"].shape}')
print(f'  labels         : {sample_batch["label"].shape}')

---
## Cell 8 — Training & Evaluation Helper Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    """
    Train model for one epoch.
    Returns: avg_loss, accuracy
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            labels         = labels
        )
        loss   = outputs.loss
        logits = outputs.logits

        loss.backward()
        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate_model(model, loader, device):
    """
    Evaluate model on a DataLoader (no gradient updates).
    Returns: avg_loss, accuracy, predictions, true_labels
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels      = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss   = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds       = torch.argmax(logits, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


def compute_metrics(y_true, y_pred, label=''):
    """Compute and print Accuracy, Precision, Recall, F1."""
    acc  = accuracy_score(y_true,  y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true,    y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true,        y_pred, average='weighted', zero_division=0)
    if label:
        print(f'\n--- {label} ---')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    return {'accuracy': round(acc,4), 'precision': round(prec,4),
            'recall': round(rec,4), 'f1': round(f1,4)}


def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """Plot a labelled confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'})
    plt.title(title, fontsize=12, fontweight='bold')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()


def plot_training_curves(history, title='Training Curves'):
    """Plot loss and accuracy over epochs for train and validation."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history['train_loss'], 'o-', color='#1a4e8a', label='Train Loss')
    axes[0].plot(epochs, history['val_loss'],   's--',color='#b7410e', label='Val Loss')
    axes[0].set_title('Loss over Epochs',     fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], 'o-', color='#1a4e8a', label='Train Acc')
    axes[1].plot(epochs, history['val_acc'],   's--',color='#2d6a4f', label='Val Acc')
    axes[1].set_title('Accuracy over Epochs', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05); axes[1].legend()

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


def run_experiment(model, train_loader, val_loader, test_loader,
                   epochs, lr, device, exp_name):
    """
    Full training loop for one experiment.
    Uses AdamW optimizer and linear warmup scheduler.
    Returns: test metrics dict and training history.
    """
    total_steps   = len(train_loader) * epochs
    warmup_steps  = total_steps // 10   # 10% warmup

    optimizer  = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=lr, weight_decay=0.01)
    scheduler  = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps
    )

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_val_acc  = 0.0
    patience      = 2    # early stopping patience
    patience_ctr  = 0

    print(f'\nStarting {exp_name} | Epochs={epochs} | LR={lr}')
    print(f'Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    print('-' * 60)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        vl_loss, vl_acc, _, _ = evaluate_model(model, val_loader, device)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        elapsed = time.time() - t0
        print(f'Epoch {epoch}/{epochs} | '
              f'Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | '
              f'Val Loss={vl_loss:.4f} Acc={vl_acc:.4f} | '
              f'Time={elapsed:.1f}s')

        # Early stopping check
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping triggered at epoch {epoch}.')
                break

    # Final test evaluation
    _, _, preds, labels = evaluate_model(model, test_loader, device)
    metrics = compute_metrics(labels, preds, label=f'{exp_name} — Test Results')
    print('\nClassification Report:')
    print(classification_report(labels, preds,
          target_names=['Negative','Positive'], zero_division=0))
    plot_confusion_matrix(labels, preds, title=f'Confusion Matrix — {exp_name}')
    plot_training_curves(history, title=f'Training Curves — {exp_name}')

    return metrics, history


print('All helper functions defined successfully.')

---
## Cell 9 — Experiment 1: Freeze All BERT Layers

All BERT encoder weights are **frozen**. Only the final classification head is trained.

| | |
|---|---|
| **Trainable** | Classification head only |
| **Frozen** | All 12 BERT encoder layers |
| **Benefit** | Very fast, no risk of catastrophic forgetting |
| **Limitation** | BERT representations not adapted to this specific task |

In [ ]:
EPOCHS = 3
LR     = 2e-5

print('Loading BERT for Experiment 1 (Freeze All BERT Layers)...')
model_exp1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# Freeze ALL BERT encoder parameters
for name, param in model_exp1.named_parameters():
    if 'classifier' not in name:   # keep classifier unfrozen
        param.requires_grad = False

frozen_params   = sum(p.numel() for p in model_exp1.parameters() if not p.requires_grad)
trainable_params= sum(p.numel() for p in model_exp1.parameters() if p.requires_grad)
print(f'Frozen params    : {frozen_params:,}')
print(f'Trainable params : {trainable_params:,}')

model_exp1 = model_exp1.to(DEVICE)

metrics_exp1, history_exp1 = run_experiment(
    model_exp1, train_loader, val_loader, test_loader,
    epochs=EPOCHS, lr=LR, device=DEVICE,
    exp_name='Exp 1: Frozen BERT'
)

---
## Cell 10 — Experiment 2: Fine-Tune Last 2 BERT Layers

Only the **last 2 encoder layers** and the classification head are trainable.
BERT has 12 encoder layers (layer.0 to layer.11).

| | |
|---|---|
| **Trainable** | Layers 10 & 11 + classifier |
| **Frozen** | Layers 0–9 + embeddings |
| **Benefit** | Good balance — task-specific adaptation without full retraining |
| **Limitation** | Slower than Exp 1 but faster than full fine-tuning |

In [ ]:
print('Loading BERT for Experiment 2 (Fine-tune Last 2 Layers)...')
model_exp2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# Freeze all parameters first
for param in model_exp2.parameters():
    param.requires_grad = False

# Unfreeze last 2 encoder layers (layer 10 and layer 11)
for name, param in model_exp2.named_parameters():
    if ('encoder.layer.10' in name or
        'encoder.layer.11' in name or
        'classifier'       in name or
        'pooler'           in name):
        param.requires_grad = True

frozen_params    = sum(p.numel() for p in model_exp2.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
print(f'Frozen params    : {frozen_params:,}')
print(f'Trainable params : {trainable_params:,}')

model_exp2 = model_exp2.to(DEVICE)

metrics_exp2, history_exp2 = run_experiment(
    model_exp2, train_loader, val_loader, test_loader,
    epochs=EPOCHS, lr=LR, device=DEVICE,
    exp_name='Exp 2: Last 2 Layers'
)

---
## Cell 11 — Experiment 3: Full BERT Fine-Tuning

**All layers** of BERT are trainable — full fine-tuning on our dataset.

| | |
|---|---|
| **Trainable** | All 12 layers + embeddings + classifier |
| **Frozen** | Nothing |
| **Benefit** | Best task-specific adaptation and highest potential accuracy |
| **Limitation** | Slowest, needs more data to avoid overfitting |

In [ ]:
print('Loading BERT for Experiment 3 (Full Fine-Tuning)...')
model_exp3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# All parameters trainable — no freezing
for param in model_exp3.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model_exp3.parameters() if p.requires_grad)
print(f'Trainable params : {trainable_params:,}  (all layers unfrozen)')

model_exp3 = model_exp3.to(DEVICE)

metrics_exp3, history_exp3 = run_experiment(
    model_exp3, train_loader, val_loader, test_loader,
    epochs=EPOCHS, lr=LR, device=DEVICE,
    exp_name='Exp 3: Full Fine-Tuning'
)

---
## Cell 12 — Bonus: DistilBERT with LR Scheduler + Early Stopping

DistilBERT is a lighter, faster version of BERT — 40% fewer parameters, 60% faster,
while retaining ~97% of BERT's performance on most tasks.

In [ ]:
DISTIL_MODEL = 'distilbert-base-uncased'

print(f'Loading DistilBERT tokenizer and model: {DISTIL_MODEL} ...')
distil_tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)

# Create DistilBERT datasets with its own tokenizer
distil_train_ds = SentimentDataset(X_train, y_train, distil_tokenizer, MAX_LEN)
distil_val_ds   = SentimentDataset(X_val,   y_val,   distil_tokenizer, MAX_LEN)
distil_test_ds  = SentimentDataset(X_test,  y_test,  distil_tokenizer, MAX_LEN)

distil_train_loader = DataLoader(distil_train_ds, batch_size=BATCH_SIZE, shuffle=True)
distil_val_loader   = DataLoader(distil_val_ds,   batch_size=BATCH_SIZE, shuffle=False)
distil_test_loader  = DataLoader(distil_test_ds,  batch_size=BATCH_SIZE, shuffle=False)

model_distil = AutoModelForSequenceClassification.from_pretrained(
    DISTIL_MODEL, num_labels=2
)
trainable_params = sum(p.numel() for p in model_distil.parameters())
print(f'DistilBERT params : {trainable_params:,}')
print(f'BERT params       : {sum(p.numel() for p in model_exp3.parameters()):,}')
print(f'Parameter saving  : {(1 - trainable_params/sum(p.numel() for p in model_exp3.parameters()))*100:.0f}%')

model_distil = model_distil.to(DEVICE)

metrics_distil, history_distil = run_experiment(
    model_distil, distil_train_loader, distil_val_loader, distil_test_loader,
    epochs=EPOCHS, lr=LR, device=DEVICE,
    exp_name='Bonus: DistilBERT'
)

---
## Cell 13 — Final Comparison of All Experiments

In [ ]:
# Build comparison table
comparison = pd.DataFrame([
    {'Experiment': 'Exp 1: Frozen BERT',      **metrics_exp1},
    {'Experiment': 'Exp 2: Last 2 Layers',    **metrics_exp2},
    {'Experiment': 'Exp 3: Full Fine-Tuning', **metrics_exp3},
    {'Experiment': 'Bonus: DistilBERT',       **metrics_distil},
])

print('=' * 70)
print('FINAL EXPERIMENT COMPARISON')
print('=' * 70)
print(comparison.to_string(index=False))
print('=' * 70)

best_exp = comparison.loc[comparison['f1'].idxmax(), 'Experiment']
best_f1  = comparison['f1'].max()
print(f'\nBest Experiment : {best_exp}')
print(f'Best F1 Score   : {best_f1}')

# Bar chart comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
x     = np.arange(len(comparison))
width = 0.20
colors = ['#1a4e8a','#2d6a4f','#b7410e','#7b3fa0']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    bars = ax.bar(x + i*width, comparison[metric], width=width,
                  label=metric.capitalize(), color=color,
                  edgecolor='white', alpha=0.9)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(comparison['Experiment'], rotation=12, ha='right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('All Experiments — Metric Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(0.5, color='red', linewidth=0.8, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# F1 Score heatmap
pivot = comparison[['Experiment','f1']].set_index('Experiment').T
plt.figure(figsize=(10, 2))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, linecolor='white', vmin=0.3, vmax=1.0,
            annot_kws={'size': 12, 'weight': 'bold'})
plt.title('F1 Score Heatmap — All Experiments', fontsize=11, fontweight='bold')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Cell 14 — Live Inference Demo

In [ ]:
def predict(text, model, tokenizer, device, max_len=128):
    """
    Predict sentiment for a single raw text input.
    Returns: label string and confidence score.
    """
    model.eval()
    cleaned = clean_text(text)
    enc = tokenizer.encode_plus(
        cleaned,
        max_length            = max_len,
        padding               = 'max_length',
        truncation            = True,
        return_attention_mask = True,
        return_tensors        = 'pt'
    )
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1).item()
        conf    = probs[0][pred].item()

    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    return label, conf


test_inputs = [
    'This movie was absolutely fantastic. I loved every single moment of it!',
    'Terrible film. Complete waste of time. The worst movie I have ever seen.',
    'An average film. Some good moments but nothing particularly memorable.',
    'Not a good movie at all. The acting was really unconvincing throughout.',
    'Surprisingly enjoyable! Exceeded all my expectations. Highly recommended!',
]

print('Live Sentiment Prediction — Best Model (Exp 3: Full Fine-Tuning)')
print('=' * 65)
for text in test_inputs:
    label, conf = predict(text, model_exp3, tokenizer, DEVICE)
    icon = 'POSITIVE' if label == 'POSITIVE' else 'NEGATIVE'
    print(f'  Input      : {text[:60]}')
    print(f'  Prediction : {icon}  (confidence: {conf:.2%})')
    print()

---
## Summary of Findings

### Experiment Comparison

| Experiment | Trainable Layers | Speed | Expected F1 | Best For |
|------------|-----------------|-------|-------------|----------|
| Exp 1: Frozen BERT | Classifier only | Fastest | ~0.65–0.70 | Quick baseline |
| Exp 2: Last 2 Layers | Layers 10–11 + classifier | Medium | ~0.70–0.80 | Balanced approach |
| Exp 3: Full Fine-Tuning | All 12 layers | Slowest | ~0.80–0.90 | Best accuracy |
| Bonus: DistilBERT | All (smaller model) | Fast | ~0.75–0.85 | Speed + accuracy |

### Key Insights

**Why TF Fine-Tuning beats Frozen BERT:**
BERT was pre-trained on general text (Wikipedia + BooksCorpus). Fine-tuning allows the model to adapt its internal representations specifically to movie review sentiment — the deeper layers learn task-specific features that a frozen model cannot.

**Why Last 2 Layers is a good tradeoff:**
Lower BERT layers capture general syntactic features (useful for any task). Upper layers capture task-specific semantic features. Unfreezing only the top layers gives most of the fine-tuning benefit at a fraction of the computational cost.

**DistilBERT advantage:**
40% fewer parameters, 60% faster inference, retains ~97% of BERT accuracy. Ideal for production systems where speed and memory matter.

**AdamW + Linear Warmup Scheduler:**
AdamW decouples weight decay from the gradient update (unlike standard Adam), reducing overfitting. Linear warmup prevents large gradient updates in early epochs when the model weights are still random.

### Best Configuration
> **Full fine-tuning (Exp 3)** achieves the highest accuracy on this task. For production or resource-constrained environments, **DistilBERT** offers the best speed-accuracy tradeoff. Always use **TF-IDF** vocabulary insights to guide your understanding of what the model is learning.